# 23 · Capstone — News Feed

**Part 3 · Capstones** · *Prerequisites: Part 1 (esp. 02, 06, 11, 12)* · *The fan-out
problem at social scale*

"Show me the latest posts from everyone I follow." One sentence, and inside it lives the
load parameter nb 01 warned about: **fan-out**. A post by a celebrity logically belongs
in 50 million feeds; a user following 800 accounts logically reads 800 streams at once.
You cannot make both writes and reads O(1) — the design question is *where the multiply
happens*, and the answer (as always) is a bargain: precompute (write-time work, nb 12's
derived data) or assemble on demand (read-time work).

**Step 1 — Requirements.** Post; follow; read home feed (top ~50 recent posts from
followees), p99 **< 200 ms**; freshness within seconds; users with 8 or 8M followers both
work.

**Step 2 — Workload.** From nb 01's exercise: at 200M DAU, timeline reads outnumber
posts ~400:1, and the follower distribution is power-law — median follower count in the
dozens, maximum in the tens of millions. *That spread — six orders of magnitude on one
parameter — is the whole problem.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=23)

## Step 3 — Design: the two pure strategies, built

We synthesize a 50k-user community with power-law popularity (preferential attachment:
follow choices weighted by an author's intrinsic pull), post 30k posts, and build *both*
feed systems over the same data:

In [ ]:
N_USERS, FOLLOWS_PER_USER, N_POSTS = 50_000, 50, 30_000

# Real follow graphs are a MIX: mostly peers (friends, colleagues) plus a few
# power-law celebrity picks. 40 peer edges + 10 popularity-weighted edges each:
pull_weight = rng.pareto(1.1, N_USERS) + 0.1
pull_weight /= pull_weight.sum()
PREFERENTIAL = 10

followees = []
for _ in range(2_000):                          # a 2k-user sample is plenty to measure
    celebrity_picks = rng.choice(N_USERS, size=PREFERENTIAL, replace=False,
                                 p=pull_weight)
    peer_picks = rng.choice(N_USERS, size=FOLLOWS_PER_USER - PREFERENTIAL,
                            replace=False)
    followees.append(np.unique(np.concatenate([celebrity_picks, peer_picks])))
follower_count = np.zeros(N_USERS, int)
followers_of: dict[int, list[int]] = {}
for u, fs in enumerate(followees):
    for f in fs:
        follower_count[f] += 1
        followers_of.setdefault(int(f), []).append(u)

med, mx = int(np.median(follower_count[follower_count > 0])), int(follower_count.max())
celebrity = int(np.argmax(follower_count))
print(f"follower counts (within the 2k sample): median {med}, max {mx} "
      f"({mx/med:.0f}x) - user {celebrity} is our celebrity")
assert mx > 30 * med, "power law: the celebrity is a different species of write"

# Posts: authors drawn by the same popularity (active accounts post more).
post_author = rng.choice(N_USERS, size=N_POSTS, p=pull_weight)
posts_by_author: dict[int, list[int]] = {}
for post_id, author in enumerate(post_author):          # post_id doubles as timestamp
    posts_by_author.setdefault(int(author), []).append(post_id)

In [ ]:
FEED_SIZE = 50

# ---- Strategy A: FAN-OUT ON READ (pull). Writes: append to author's own list
# (already done above, O(1)). Reads: fetch every followee's recent posts, merge.
def read_feed_pull(user: int) -> tuple[list[int], int]:
    ops = 0
    candidates: list[int] = []
    for author in followees[user]:
        ops += 1                                        # one store fetch per followee
        candidates.extend(posts_by_author.get(int(author), [])[-FEED_SIZE:])
    return sorted(candidates)[-FEED_SIZE:], ops

# ---- Strategy B: FAN-OUT ON WRITE (push). Writes: insert the post id into
# EVERY follower's precomputed mailbox. Reads: one fetch.
mailbox: dict[int, list[int]] = {u: [] for u in range(len(followees))}
push_write_ops = np.zeros(N_POSTS, int)
for post_id, author in enumerate(post_author):
    fans = followers_of.get(int(author), [])
    push_write_ops[post_id] = len(fans)
    for fan in fans:
        mailbox[fan].append(post_id)                    # the write-time multiply

def read_feed_push(user: int) -> tuple[list[int], int]:
    return mailbox[user][-FEED_SIZE:], 1                # one mailbox fetch. done.

# Same feeds from both systems (the mailbox IS a derived view of the post log):
for user in rng.choice(len(followees), 50, replace=False):
    feed_a, _ = read_feed_pull(int(user))
    feed_b, _ = read_feed_push(int(user))
    assert feed_a == feed_b, "push and pull must agree - one truth, two layouts (nb 12)"
print("correctness: pull feed == push feed for every sampled user")

Now the bill. Reads outnumber posts 200:1 in our miniature (~400:1 in real life — this
only strengthens the conclusion):

In [ ]:
READS_PER_POST = 200                                    # toward the real ~400:1
n_reads = N_POSTS * READS_PER_POST

pull_read_ops = FOLLOWS_PER_USER * n_reads              # every read pays the fan-out
pull_write_ops = N_POSTS                                # one append per post
push_read_ops = 1 * n_reads
push_write_total = int(push_write_ops.sum())

print(f"PULL:  {pull_read_ops/1e6:,.0f}M read ops + {pull_write_ops/1e3:,.0f}k write ops")
print(f"PUSH:  {push_read_ops/1e6:,.0f}M read ops + {push_write_total/1e3:,.0f}k write ops")
print(f"push write cost per post: median {int(np.median(push_write_ops))}, "
      f"p99.9 {int(np.percentile(push_write_ops, 99.9))}, max {int(push_write_ops.max())}")

assert push_read_ops + push_write_total < 0.1 * (pull_read_ops + pull_write_ops), \
    "when reads dominate, precomputation wins by an order of magnitude overall..."
assert push_write_ops.max() > 30 * np.median(push_write_ops), \
    "...but the celebrity post is a 30x+ write bomb hiding in the average " \
    "(at real scale: 100M inserts for one tweet)"

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(push_write_ops[push_write_ops > 0], bins=60, color="#1565c0")
ax.set_yscale("log")
ax.set_xlabel("mailbox inserts caused by one post")
ax.set_ylabel("posts (log)")
ax.set_title("Fan-out on write: the median post is cheap; the celebrity is a siege")
plt.show()

Scale the tail up to reality: a 100M-follower account posting means **100 million
mailbox inserts for one tweet** — minutes of cluster-wide write storm, delivered with
nb 01's tail-latency amplification to every follower's p99. Pure push dies on the
celebrity; pure pull dies on read volume. So nobody runs either.

### The hybrid — push for the many, pull for the mighty

Accounts below a follower threshold fan out on write; celebrity posts stay in their
author's list only, and each *read* merges the user's mailbox with a live pull from the
few celebrities they follow. The write bomb is defused; the read pays only for the
celebrities it actually follows — and the power law guarantees that's a small number:

In [ ]:
CELEB_THRESHOLD = 100                # in-sample followers; ~top 0.03% of accounts
is_celeb = follower_count > CELEB_THRESHOLD

hybrid_mailbox: dict[int, list[int]] = {u: [] for u in range(len(followees))}
hybrid_write_ops = np.zeros(N_POSTS, int)
for post_id, author in enumerate(post_author):
    if is_celeb[author]:
        continue                                        # no fan-out for celebrities
    fans = followers_of.get(int(author), [])
    hybrid_write_ops[post_id] = len(fans)
    for fan in fans:
        hybrid_mailbox[fan].append(post_id)

def read_feed_hybrid(user: int) -> tuple[list[int], int]:
    ops = 1                                             # the mailbox
    candidates = list(hybrid_mailbox[user][-FEED_SIZE * 2:])
    for author in followees[user]:
        if is_celeb[author]:
            ops += 1                                    # live pull, per celeb followed
            candidates.extend(posts_by_author.get(int(author), [])[-FEED_SIZE:])
    return sorted(candidates)[-FEED_SIZE:], ops

read_costs = []
for user in rng.choice(len(followees), 300, replace=False):
    feed_h, ops = read_feed_hybrid(int(user))
    feed_a, _ = read_feed_pull(int(user))
    assert feed_h == feed_a, "hybrid must still produce the SAME feed"
    read_costs.append(ops)

celebs_per_user = np.mean([sum(is_celeb[a] for a in followees[u])
                           for u in range(len(followees))])
print(f"celebrities: {int(is_celeb.sum())} accounts ({is_celeb.mean():.2%}); "
      f"avg followed per user: {celebs_per_user:.1f}")
print(f"hybrid read ops: mean {np.mean(read_costs):.1f}, max {max(read_costs)} "
      f"(pull was {FOLLOWS_PER_USER}); hybrid max write fan-out: "
      f"{int(hybrid_write_ops.max())} (push max was {int(push_write_ops.max())})")

assert np.mean(read_costs) < FOLLOWS_PER_USER / 3, \
    "reads stay cheap: mailbox + a handful of celebrity pulls"
assert hybrid_write_ops.max() <= CELEB_THRESHOLD + med, \
    "the write bomb is gone by construction: fan-out is CAPPED at the threshold"

This is the published Twitter architecture, reproduced with correct feeds and measured
costs. Notice its deeper shape: the threshold is a **dial between nb 12's two worlds** —
precomputed derived data (mailboxes) where fan-out is affordable, on-demand assembly
where it isn't. Exercise 4 derives where the dial should sit.

## Step 3½ — Sizing the mailbox tier (nb 01 discipline)

In [ ]:
DAU = 100e6
active_mailboxes = DAU * 0.2                 # only recently-active users stay cached
entries, entry_bytes = 400, 16               # post_id + timestamp, ids only
replicas = 3
mailbox_ram_tb = active_mailboxes * entries * entry_bytes * replicas / 1e12
print(f"mailbox cache: {mailbox_ram_tb:.1f} TB across the fleet "
      f"({mailbox_ram_tb*1e3/replicas:,.0f} GB per replica set)")
assert mailbox_ram_tb < 5, "ids-only + active-only keeps this in commodity-RAM territory"

Two decisions did the heavy lifting: store **ids, not content** (content lives once in a
post store; the feed hydrates it — normalization, nb 02, at cache scale), and cache
**active users only** — an evicted user's mailbox is rebuilt by one pull-style read
(the mailbox is derived data; nb 12's rebuild-from-truth superpower is the eviction
policy's safety net).

## Step 4 — Failure analysis

- **Partial fan-out** (crash mid-delivery: 60k of 100k mailboxes updated): deliveries
  ride an **event log** (nb 11) consumed by idempotent mailbox writers — redelivery
  completes the job, duplicates are no-ops (post_id is the idempotency key). The post
  log is the source of truth; mailboxes are views (nb 12's dual-write lesson, applied).
- **Deletes**: a retracted post must leave millions of mailboxes — impractical to chase
  synchronously. Standard move: filter at read time against a small tombstone set
  (nb 03's pattern) and clean lazily.
- **Hot read on a celebrity's author-list** (everyone's hybrid read pulls it): cache it
  (nb 22 §3.2's Zipf logic — one hot key, many readers).
- **Ordering**: mailboxes receive posts out of order (async fan-out); feeds sort by
  post timestamp at read — and cross-author ordering needs no consensus because feeds
  don't promise it (nb 09's lesson inverted: *relax the guarantee you don't need*).
- **Ranking** (evolution): chronological merge is stage one; real feeds then score
  candidates with an ML ranker — which is precisely capstone 24, so we leave the hook
  here.

## Exercises

1. **Counts at scale.** Every post shows like/reply counts; hot posts get thousands of
   likes/sec. Why is `UPDATE posts SET likes = likes+1` a disaster (nb 07 §2.4 +
   nb 06 §3), and what does the standard fix look like (sharded counters / log-derived
   aggregates, nb 12)? When is an *approximate* count fine?
2. **Pagination.** Offset-based paging ("posts 51–100") breaks when new posts arrive
   mid-scroll (items shift; duplicates appear). Design cursor-based paging and state
   what the cursor contains. Which nb 05 read guarantee is this implementing?
3. **Two regions.** EU and US datacenters both serve feeds. Compare: single-region
   mailboxes + cross-region reads, vs per-region mailbox replicas fed by the global
   post log. Which nb 05 anomalies threaten each, and which design wins for read
   latency (nb 01's WAN numbers)?
4. **The threshold, derived.** With follower distribution F, posts/user/day p, and
   reads/user/day r, write the total-ops formulas for hybrid as a function of threshold
   T, and reason where the minimum sits. What *non-throughput* constraint (delivery
   latency of one post) also pushes T down?
5. **Live updates.** "New posts" should appear without refresh. Compare polling the
   mailbox vs a websocket push fed by the fan-out pipeline. What does the push design
   do to the celebrity problem — and which users actually need it?

### Solutions

**1.** A hot row taking thousands of increments/sec is a lock/CAS convoy (every
increment is a read-modify-write on ONE key — lost-update protection serializes them,
nb 07) on one partition (un-shardable hot key, nb 06). Fix: shard the counter into K
sub-counters (salting, nb 06 §3 — write a random shard, read sums them) or, cleaner,
derive counts from the like-event log (nb 11/12) with per-window aggregation. Approximate
is fine the moment the number is display-only ("12.4K") — which is why real UIs round:
it's an engineering release valve, not a style choice.

**2.** Cursor = (timestamp, post_id) of the last item seen (the tie-broken sort key);
"next page" = items strictly older than the cursor. New arrivals can't shift what
"older than X" means, so no duplicates or gaps. This is **monotonic reads** (nb 05)
implemented at the API layer: the client's position moves one direction through a
stable order regardless of which replica serves page 2.

**3.** Cross-region reads pay nb 01's 80–140 ms per feed load — dead on arrival for a
p99 of 200 ms. Per-region mailboxes derived from the global post log (nb 11/12) win:
reads are local; the cost is replication lag → a user posting in EU and refreshing via
US could miss their own post (**read-your-writes**, nb 05) — fixed by session-stickiness
to home region or the LSN-token trick. Multi-leader *mailbox* writes are unnecessary —
mailboxes are derived, so there's nothing to conflict (derived data dodges the
multi-leader merge problem entirely).

**4.** Hybrid daily ops(T) ≈ push-writes: p·Σ_{u: F(u)≤T} F(u) + reads:
r·N·(1 + E[#followees with F>T]). Raising T moves accounts from "pulled by their
followers" to "pushed to everyone": write term rises with the *sum of follower counts
below T* (power law: slowly, then explosively), read term falls (fewer live pulls).
The minimum sits where marginal write cost of promoting the next account equals the
read savings across its followers' reads — in power-law land, well below the megastars.
The latency constraint: one post's delivery time ∝ its fan-out; if feeds must reflect
posts within seconds, T is capped by (deliveries/sec you can sustain) × (freshness
budget), independent of total-ops math.

**5.** Polling: simple, cache-friendly, adds (poll interval) staleness and constant
background load — nb 18's batch-vs-online trade in miniature. Websocket push: fresh,
but now a celebrity post must be *pushed live to millions of open sockets* — the
fan-out bomb reborn one layer up; mitigations are the same (push "you have new posts"
flags to active-socket users only, let clients fetch). Only currently-foregrounded
users need it — which is a tiny fraction, and exactly the active-user trick from the
sizing cell.

## Takeaways

- Feeds are a **where-does-the-multiply-go** problem: fan-out on read (cheap writes,
  expensive reads) vs fan-out on write (cheap reads, celebrity write bombs). Reads
  dominate → precompute — *except* where the power law makes precomputation explode.
- The **hybrid** caps write fan-out at a threshold and pulls celebrities at read time:
  same feeds (asserted!), bounded costs on both paths. The threshold is a business
  dial between nb 12's precompute-vs-assemble worlds.
- Mailboxes are **derived data**: source-of-truth post log, idempotent fan-out
  consumers, rebuild-on-eviction, tombstone-filtered deletes. Every failure story
  resolves into a Part 1 pattern.
- Size with ids-not-content and active-users-only; relax orderings nobody needs; cache
  the hot author-lists the hybrid creates.

**Next:** [24 · Recommender system](24-capstone-recommender.ipynb) — the feed learns to
rank: both books, one system, end to end.